# Additional End of week Exercise - week 2

Now use everything you've learned from Week 2 to build a full prototype for the technical question/answerer you built in Week 1 Exercise.

This should include a Gradio UI, streaming, use of the system prompt to add expertise, and the ability to switch between models. Bonus points if you can demonstrate use of a tool!

If you feel bold, see if you can add audio input so you can talk to it, and have it respond with audio. ChatGPT or Claude can help you, or email me if you have questions.

I will publish a full solution here soon - unless someone beats me to it...

There are so many commercial applications for this, from a language tutor, to a company onboarding solution, to a companion AI to a course (like this one!) I can't wait to see your results.

## Split run version (block-by-block)
Run the following cells in order.

- Cell 3 now installs or upgrades dependencies (`gradio`, `python-dotenv`, `openai`) in the active notebook kernel.
- If packages are installed/upgraded for the first time, rerun Cell 3 once more.


In [ ]:
# 1) Dependency bootstrap + imports + setup + model catalog

import sys
import subprocess


def ensure_package(module_name, package_name=None, upgrade=False):
    package = package_name or module_name

    # If module imports and upgrade is not requested, keep current install.
    if not upgrade:
        try:
            __import__(module_name)
            return
        except ImportError:
            pass

    # Install or upgrade package in the active notebook kernel environment.
    cmd = [sys.executable, "-m", "pip", "install"]
    if upgrade:
        cmd.append("--upgrade")
    cmd.append(package)
    subprocess.check_call(cmd)


# Keep core notebook dependencies up-to-date.
ensure_package("gradio", upgrade=True)
ensure_package("dotenv", "python-dotenv", upgrade=True)
ensure_package("openai", upgrade=True)

import os
import json
import pydoc
import tempfile
from typing import Dict, List, Any, Tuple

import gradio as gr
from dotenv import load_dotenv
from openai import OpenAI

print(f"Gradio version: {gr.__version__}")

load_dotenv(override=True)

openai_api_key = os.getenv("OPENAI_API_KEY")
openrouter_api_key = os.getenv("OPENROUTER_API_KEY")

openai_client = OpenAI(api_key=openai_api_key) if openai_api_key else None
openrouter_client = (
    OpenAI(api_key=openrouter_api_key, base_url="https://openrouter.ai/api/v1")
    if openrouter_api_key
    else None
)

# Ollama can be used without a cloud key if running locally.
ollama_client = OpenAI(base_url="http://localhost:11434/v1", api_key="ollama")


def is_ollama_available() -> bool:
    try:
        ollama_client.models.list()
        return True
    except Exception:
        return False


OLLAMA_AVAILABLE = is_ollama_available()


def build_model_catalog() -> Dict[str, Dict[str, Any]]:
    catalog = {}

    if openai_client:
        catalog["OpenAI - gpt-4.1-mini"] = {
            "client": openai_client,
            "model": "gpt-4.1-mini",
            "supports_tools": True,
            "supports_audio": True,
        }
        catalog["OpenAI - gpt-4o-mini"] = {
            "client": openai_client,
            "model": "gpt-4o-mini",
            "supports_tools": True,
            "supports_audio": True,
        }

    if openrouter_client:
        catalog["OpenRouter - openai/gpt-4o-mini"] = {
            "client": openrouter_client,
            "model": "openai/gpt-4o-mini",
            "supports_tools": True,
            "supports_audio": False,
            # Keep responses within budget for credit-limited OpenRouter keys.
            "max_tokens": 1200,
        }

    if OLLAMA_AVAILABLE:
        catalog["Ollama - llama3.2"] = {
            "client": ollama_client,
            "model": "llama3.2",
            "supports_tools": False,
            "supports_audio": False,
        }

    return catalog


MODEL_CATALOG = build_model_catalog()
MODEL_NAMES = list(MODEL_CATALOG.keys())

if not MODEL_NAMES:
    raise RuntimeError(
        "No models available. Set OPENAI_API_KEY or OPENROUTER_API_KEY, or start Ollama locally."
    )


def provider_status_markdown() -> str:
    lines = []
    lines.append("### Provider and Model Status")
    lines.append(f"- OPENAI_API_KEY: {'Available' if openai_api_key else 'Missing'}")
    lines.append(f"- OPENROUTER_API_KEY: {'Available' if openrouter_api_key else 'Missing'}")
    lines.append(f"- Ollama reachable: {'Yes' if OLLAMA_AVAILABLE else 'No'}")

    lines.append("\n**Loaded models**")
    for model_name in MODEL_NAMES:
        cfg = MODEL_CATALOG[model_name]
        tool_status = "Yes" if cfg["supports_tools"] else "No"
        audio_status = "Yes" if cfg["supports_audio"] else "No"
        lines.append(f"- {model_name} | tools: {tool_status} | voice out: {audio_status}")

    return "\n".join(lines)


DEFAULT_SYSTEM_PROMPT = """
You are a senior technical mentor and software engineer.
Give concise, correct, practical answers to technical questions.
When useful, include short examples and common pitfalls.
If the user asks for code explanation, explain step-by-step and include why it matters.
If you are uncertain, say what is uncertain and how to verify.
Respond in markdown.
""".strip()

print(provider_status_markdown())

In [ ]:
# 2) Tool definitions

def get_python_reference(symbol: str) -> str:
    """Return a local Python help snippet using pydoc."""
    symbol = (symbol or "").strip()
    if not symbol:
        return "No symbol provided."
    try:
        doc = pydoc.render_doc(symbol, title="Help on %s")
        lines = doc.splitlines()
        return "\n".join(lines[:80])
    except Exception:
        return f"No local Python docs found for '{symbol}'."


python_reference_tool = {
    "type": "function",
    "function": {
        "name": "get_python_reference",
        "description": "Get local Python docs for a symbol such as json.dumps or itertools.groupby.",
        "parameters": {
            "type": "object",
            "properties": {
                "symbol": {
                    "type": "string",
                    "description": "Python symbol name, for example: collections.Counter",
                }
            },
            "required": ["symbol"],
            "additionalProperties": False,
        },
    },
}

TOOLS = [python_reference_tool]


def handle_tool_calls(tool_calls) -> List[Dict[str, str]]:
    responses = []
    for tool_call in tool_calls:
        name = tool_call.function.name
        arguments = json.loads(tool_call.function.arguments or "{}")

        if name == "get_python_reference":
            result = get_python_reference(arguments.get("symbol", ""))
        else:
            result = f"Unknown tool call: {name}"

        responses.append(
            {
                "role": "tool",
                "tool_call_id": tool_call.id,
                "content": result,
            }
        )
    return responses

In [ ]:
# 3) Audio helpers

def transcribe_audio(audio_path: str) -> str:
    """Transcribe microphone audio if OpenAI key exists."""
    if not audio_path:
        return ""
    if not openai_client:
        return ""

    with open(audio_path, "rb") as audio_file:
        transcript = openai_client.audio.transcriptions.create(
            model="gpt-4o-mini-transcribe",
            file=audio_file,
        )
    return transcript.text


def speak_text(text: str) -> str:
    """Convert assistant text reply to mp3, return file path."""
    if not text or not openai_client:
        return None

    speech = openai_client.audio.speech.create(
        model="gpt-4o-mini-tts",
        voice="alloy",
        input=text,
    )

    with tempfile.NamedTemporaryFile(delete=False, suffix=".mp3") as tmp:
        tmp.write(speech.content)
        return tmp.name

In [ ]:
# 4) Chat core

def normalize_history(history):
    """Convert Gradio history into OpenAI-style message dicts."""
    clean = []
    for item in history or []:
        # Dict style: {"role": ..., "content": ...}
        if isinstance(item, dict):
            role = item.get("role")
            content = item.get("content")
            if role in ("user", "assistant"):
                text = _extract_text(content)
                if text.strip():
                    clean.append({"role": role, "content": text})
            continue

        # Object style (e.g., gr.ChatMessage): has role/content attributes.
        role = getattr(item, "role", None)
        content = getattr(item, "content", None)
        if role in ("user", "assistant"):
            text = _extract_text(content)
            if text.strip():
                clean.append({"role": role, "content": text})
            continue

        # Older tuple style: (user_text, assistant_text)
        if isinstance(item, (list, tuple)) and len(item) == 2:
            user_text, assistant_text = item
            user_text = _extract_text(user_text)
            assistant_text = _extract_text(assistant_text)
            if user_text.strip():
                clean.append({"role": "user", "content": user_text})
            if assistant_text.strip():
                clean.append({"role": "assistant", "content": assistant_text})

    return clean


def _extract_text(content) -> str:
    """Extract plain text from provider content shapes (str/list/dict)."""
    if isinstance(content, str):
        return content

    if isinstance(content, list):
        parts = []
        for item in content:
            if isinstance(item, dict):
                # OpenAI-style rich content parts
                text = item.get("text") or item.get("content")
                if isinstance(text, str):
                    parts.append(text)
        return "".join(parts)

    if isinstance(content, dict):
        text = content.get("text") or content.get("content")
        if isinstance(text, str):
            return text

    return ""


def stream_assistant_reply(
    user_message: str,
    history: List[Dict[str, str]],
    model_choice: str,
    system_prompt: str,
    use_tool: bool,
):
    cfg = MODEL_CATALOG[model_choice]
    client = cfg["client"]
    model = cfg["model"]
    supports_tools = cfg["supports_tools"]
    max_tokens = cfg.get("max_tokens")

    messages = [{"role": "system", "content": system_prompt}] + normalize_history(history)
    messages.append({"role": "user", "content": user_message})

    # Shared request options, with optional token cap for credit-limited providers.
    request_kwargs = {"model": model, "messages": messages}
    if max_tokens:
        request_kwargs["max_tokens"] = max_tokens

    # If tools are enabled, first resolve tool calls, then stream final answer.
    if use_tool and supports_tools:
        response = client.chat.completions.create(**request_kwargs, tools=TOOLS)

        while True:
            choices = getattr(response, "choices", None) or []
            if not choices:
                break

            choice = choices[0]
            if getattr(choice, "finish_reason", None) != "tool_calls":
                break

            tool_message = getattr(choice, "message", None)
            tool_calls = getattr(tool_message, "tool_calls", None) or []
            if not tool_calls:
                break

            tool_responses = handle_tool_calls(tool_calls)
            messages.append(tool_message)
            messages.extend(tool_responses)
            response = client.chat.completions.create(**request_kwargs, tools=TOOLS)

        stream = client.chat.completions.create(**request_kwargs, stream=True)
    else:
        stream = client.chat.completions.create(**request_kwargs, stream=True)

    answer = ""
    for chunk in stream:
        choices = getattr(chunk, "choices", None) or []
        if not choices:
            continue

        delta_obj = getattr(choices[0], "delta", None)
        delta = _extract_text(getattr(delta_obj, "content", None))
        if not delta:
            continue

        answer += delta
        yield answer

    # Some providers can produce empty streamed content; fallback to non-stream call.
    if not answer.strip():
        final_response = client.chat.completions.create(**request_kwargs)
        final_choices = getattr(final_response, "choices", None) or []
        if final_choices:
            final_msg = getattr(final_choices[0], "message", None)
            final_text = _extract_text(getattr(final_msg, "content", None))
            if final_text:
                yield final_text

In [ ]:
# 5) Gradio callbacks

def add_user_message(text_message, audio_path, history):
    history = history or []
    text_message = (text_message or "").strip()

    transcript = ""
    if audio_path:
        if not openai_client:
            raise gr.Error("Audio transcription requires OPENAI_API_KEY.")
        transcript = transcribe_audio(audio_path).strip()

    if text_message and transcript:
        user_text = f"{text_message}\n\n(Voice transcript) {transcript}"
    elif text_message:
        user_text = text_message
    elif transcript:
        user_text = transcript
    else:
        raise gr.Error("Provide text or audio input.")

    # Messages format expected by current Gradio Chatbot.
    history = history + [{"role": "user", "content": user_text}]
    return "", None, history


def generate_assistant(history, model_choice, system_prompt, use_tool, voice_reply):
    history = history or []
    if not history:
        yield history, None
        return

    # Support dict or object-based chat history entries.
    last = history[-1]
    if isinstance(last, dict):
        user_message = _extract_text(last.get("content"))
    else:
        user_message = _extract_text(getattr(last, "content", ""))

    if not user_message.strip():
        yield history + [{"role": "assistant", "content": "Please enter a message."}], None
        return

    prior_history = history[:-1]

    final_text = ""
    try:
        for partial in stream_assistant_reply(
            user_message=user_message,
            history=prior_history,
            model_choice=model_choice,
            system_prompt=system_prompt,
            use_tool=use_tool,
        ):
            final_text = partial
            yield prior_history + [
                {"role": "user", "content": user_message},
                {"role": "assistant", "content": partial},
            ], None
    except Exception as e:
        error_msg = f"Error while generating response: {e}"
        yield prior_history + [
            {"role": "user", "content": user_message},
            {"role": "assistant", "content": error_msg},
        ], None
        return

    if not final_text.strip():
        final_text = (
            "No response text returned by the selected model. "
            "Try another model or disable tools for this request."
        )

    audio_output = speak_text(final_text) if voice_reply else None
    yield prior_history + [
        {"role": "user", "content": user_message},
        {"role": "assistant", "content": final_text},
    ], audio_output


def clear_everything():
    return [], "", None, None

In [ ]:
# 6) Build the Gradio UI (does not launch yet)

with gr.Blocks(title="Technical Q/A Prototyper") as demo:
    gr.Markdown("# Technical Q/A Prototyper\nStreaming + model switch + expert prompt + tool calling + optional voice")

    status_md = gr.Markdown(value=provider_status_markdown())
    refresh_status = gr.Button("Refresh status")

    with gr.Row():
        model_choice = gr.Dropdown(
            choices=MODEL_NAMES,
            value=MODEL_NAMES[0],
            label="Model",
        )
        use_tool = gr.Checkbox(value=True, label="Enable Python reference tool")
        voice_reply = gr.Checkbox(value=False, label="Speak replies (OpenAI key required)")

    system_prompt = gr.Textbox(
        label="System prompt (expertise)",
        value=DEFAULT_SYSTEM_PROMPT,
        lines=6,
    )

    # This Gradio build uses messages-format data by default, without a `type` arg.
    chatbot = gr.Chatbot(height=450, label="Technical Assistant")

    with gr.Row():
        text_input = gr.Textbox(label="Ask a technical question", lines=3)
        # Keep microphone input, but also allow file upload when no mic is detected.
        audio_input = gr.Audio(
            sources=["microphone", "upload"],
            type="filepath",
            label="Voice input (microphone or upload)",
        )

    gr.Markdown(
        "If your browser says no microphone found, allow mic permissions or upload an audio file instead."
    )

    with gr.Row():
        send = gr.Button("Send", variant="primary")
        cancel = gr.Button("Cancel generation", variant="stop")
        clear = gr.Button("Clear")

    audio_output = gr.Audio(label="Assistant voice reply", autoplay=True)

    # Keep references so cancel button can stop in-flight generation events.
    send_event = send.click(
        add_user_message,
        inputs=[text_input, audio_input, chatbot],
        outputs=[text_input, audio_input, chatbot],
    ).then(
        generate_assistant,
        inputs=[chatbot, model_choice, system_prompt, use_tool, voice_reply],
        outputs=[chatbot, audio_output],
    )

    submit_event = text_input.submit(
        add_user_message,
        inputs=[text_input, audio_input, chatbot],
        outputs=[text_input, audio_input, chatbot],
    ).then(
        generate_assistant,
        inputs=[chatbot, model_choice, system_prompt, use_tool, voice_reply],
        outputs=[chatbot, audio_output],
    )

    # This stops currently running send/submit generation tasks.
    cancel.click(
        fn=None,
        inputs=None,
        outputs=None,
        cancels=[send_event, submit_event],
    )

    clear.click(
        clear_everything,
        inputs=[],
        outputs=[chatbot, text_input, audio_input, audio_output],
    )

    refresh_status.click(
        provider_status_markdown,
        inputs=[],
        outputs=[status_md],
    )

print("UI object created. Run the next cell to launch it.")

In [ ]:
# 7) Launch

# flagging_mode is not supported in this Gradio version.
demo.launch(inbrowser=True)